Import Libraries

In [ ]:
import os, re, requests, json

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from   IPython.display import Image

from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller

from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

%matplotlib inline

#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)

Import the dataset

In [ ]:
df_macro = pd.read_parquet('./Data/fred_prorcessed_daily.parquet')
col_list = ['DGS1MO', 'DGS3MO', 'DGS6MO', 'DGS1', 'DGS2', 'DGS3', 'DGS5', 'DGS7', 'DGS10', 'DGS20', 'DGS30']
df = df_macro[col_list]

Filter the time periods in df that contain NaN values

In [ ]:
df = df.loc[df.dropna(how='any').first_valid_index():]
print("Number of rows with NaN:", df.isna().any(axis=1).sum())

Number of rows with NaN: 0


In [ ]:
def add_features(file):
    features = pd.DataFrame(index=file.index).sort_index() # Create a DataFrame with index from the input DataFrame and sort it
    features['f1'] = file.Close / file.Close.ewm(span=50).mean()  # Close price divided by 50-period Exponential Moving Average (EMA)
    features['f2'] = (file.Close - file.Close.mean()) / file.Close.std()  # Z-score of closing price
    features['f3'] = file.Close.rolling(window=20).mean()  # 20-day moving average of closing price
    features['f4'] = file.Close.rolling(window=50).mean()  # 50-day moving average of closing price
    features['f5'] = file.Close.rolling(window=200).mean()  # 200-day moving average of closing price

    features['f6'] = file.Close.ewm(span=12).mean()  # 12-day EMA of closing price
    features['f7'] = file.Close.ewm(span=26).mean()  # 26-day EMA of closing price
    features['f8'] = features['f6'] - features['f7'] # MACD line (difference between 12-day and 26-day EMAs)
    features['f9'] = features['f8'].ewm(span=9).mean()  # MACD signal line (9-day EMA of MACD line)
    features['f10'] = features['f8'] - features['f9']  # MACD histogram (difference between MACD line and MACD signal line)

    features['f11'] = file.Close.rolling(window=20).mean()  # Bollinger Bands middle band (20-day moving average)
    features['f12'] = file.Close.rolling(window=20).std()  # Bollinger Bands standard deviation (20-day rolling window)
    features['f13'] = features['f11'] + 2 * features['f12']  # Bollinger Bands upper band (middle band + 2 * standard deviation)
    features['f14'] = features['f11'] - 2 * features['f12']  # Bollinger Bands lower band (middle band - 2 * standard deviation)

    features['f15'] = file.Close.diff(10)  # 10-day momentum (difference in closing price over 10 days)
    features['f16'] = file.Close.diff(50)  # 50-day momentum (difference in closing price over 50 days)

    features['f17'] = file.Close.ewm(span=14).mean()  # Keltner Channel middle band (14-day EMA)
    features['f18'] = features['f17'] + 2 * file.Close.rolling(window=14).std()  # Keltner Channel upper band (middle band + 2 * 14-day standard deviation)
    features['f19'] = features['f17'] - 2 * file.Close.rolling(window=14).std()  # Keltner Channel lower band (middle band - 2 * 14-day standard deviation)

    return features

for col in col_list:
    locals()[f'close_{col}'] = df[[col]].rename(columns={col: 'Close'})
    locals()[f'features_{col}'] = add_features(df[[col]].rename(columns={col: 'Close'}))
    locals()[f'features_{col}'] = locals()[f'features_{col}'].loc[locals()[f'features_{col}'].dropna(how='any').first_valid_index():]
    locals()[f'close_{col}'] = locals()[f'close_{col}'].loc[locals()[f'features_{col}'].dropna(how='any').first_valid_index():]

    if locals()[f'close_{col}'].isna().any(axis=1).sum() > 0:
        print("Number of rows with NaN:", f'close_{col}',locals()[f'close_{col}'].isna().any(axis=1).sum())

    if np.isinf(locals()[f'close_{col}']).any(axis=1).sum() > 0:
        print("Number of rows with infinity values:", f'close_{col}',np.isinf(locals()[f'close_{col}']).any(axis=1).sum())

    if locals()[f'features_{col}'].isna().any(axis=1).sum() > 0:
        print("Number of rows with NaN:", f'features_{col}',locals()[f'features_{col}'].isna().any(axis=1).sum())

    if np.isinf(locals()[f'features_{col}']).any(axis=1).sum() > 0:
        print("Number of rows with infinity values:", f'features_{col}',np.isinf(locals()[f'features_{col}']).any(axis=1).sum())

### Feature Selection

In [ ]:
threshold = 0.01

Ridge

In [ ]:
all_important_features_ridge = set()

for col in col_list:

    features = locals()[f'features_{col}']
    target = locals()[f'close_{col}']['Close']

    features = pd.DataFrame(StandardScaler().fit_transform(features),
                                   columns=features.columns,
                                   index=features.index)

    ridge = Ridge(solver='svd',random_state=42)

    ridge.fit(features, target)

    ridge_importance = pd.DataFrame({
        'Feature': features.columns,
        'Coefficient': ridge.coef_
    })
    ridge_importance['Abs_Coefficient'] = ridge_importance['Coefficient'].abs()
    ridge_importance = ridge_importance.sort_values('Abs_Coefficient', ascending=False)

    ridge_features = ridge_importance[ridge_importance['Abs_Coefficient'] > threshold]['Feature'].tolist()
    all_important_features_ridge.update(ridge_features)
    print(f"Ridge selected {len(ridge_features)} features for {col}:", ridge_features)

print("\n=== Final Ridge Important Features ===")
print(f"Total unique features: {len(all_important_features_ridge)}")
print(sorted(all_important_features_ridge))

Ridge selected 11 features for DGS1MO: ['f2', 'f14', 'f3', 'f11', 'f13', 'f19', 'f10', 'f17', 'f6', 'f7', 'f4']
Ridge selected 13 features for DGS3MO: ['f2', 'f14', 'f3', 'f11', 'f13', 'f18', 'f17', 'f6', 'f7', 'f19', 'f10', 'f4', 'f16']
Ridge selected 14 features for DGS6MO: ['f2', 'f14', 'f3', 'f11', 'f13', 'f18', 'f17', 'f6', 'f7', 'f19', 'f4', 'f10', 'f16', 'f15']
Ridge selected 13 features for DGS1: ['f2', 'f14', 'f11', 'f3', 'f13', 'f18', 'f17', 'f6', 'f7', 'f19', 'f10', 'f16', 'f4']
Ridge selected 11 features for DGS2: ['f2', 'f14', 'f11', 'f3', 'f13', 'f18', 'f17', 'f6', 'f7', 'f19', 'f10']
Ridge selected 12 features for DGS3: ['f2', 'f14', 'f11', 'f3', 'f13', 'f18', 'f17', 'f6', 'f7', 'f19', 'f10', 'f4']
Ridge selected 10 features for DGS5: ['f2', 'f14', 'f11', 'f3', 'f13', 'f18', 'f10', 'f17', 'f6', 'f7']
Ridge selected 6 features for DGS7: ['f2', 'f14', 'f3', 'f11', 'f13', 'f10']
Ridge selected 6 features for DGS10: ['f2', 'f14', 'f11', 'f3', 'f13', 'f10']
Ridge selected 11 

Gradient Boosting

In [ ]:
all_important_features_gb = set()

for col in col_list:
    features = locals()[f'features_{col}']
    target = locals()[f'close_{col}']['Close']

    features = pd.DataFrame(StandardScaler().fit_transform(features),
                                   columns=features.columns,
                                   index=features.index)

    gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gb.fit(features, target)

    gb_importance = pd.DataFrame({
        'Feature': features.columns,
        'Importance': gb.feature_importances_
    }).sort_values('Importance', ascending=False)

    gb_features = gb_importance[gb_importance['Importance'] > threshold]['Feature'].tolist()
    all_important_features_gb.update(gb_features)
    print(f"Gradient Boosting selected {len(gb_features)} features for {col}:",gb_features)

print("\n=== Final Gradient Boosting Important Features ===")
print(f"Total unique features: {len(all_important_features_gb)}")
print(sorted(all_important_features_gb))

Gradient Boosting selected 1 features for DGS1MO: ['f2']
Gradient Boosting selected 1 features for DGS3MO: ['f2']
Gradient Boosting selected 1 features for DGS6MO: ['f2']
Gradient Boosting selected 1 features for DGS1: ['f2']
Gradient Boosting selected 1 features for DGS2: ['f2']
Gradient Boosting selected 1 features for DGS3: ['f2']
Gradient Boosting selected 1 features for DGS5: ['f2']
Gradient Boosting selected 1 features for DGS7: ['f2']
Gradient Boosting selected 1 features for DGS10: ['f2']
Gradient Boosting selected 1 features for DGS20: ['f2']
Gradient Boosting selected 1 features for DGS30: ['f2']

=== Final Gradient Boosting Important Features ===
Total unique features: 1
['f2']


XGBoost

In [ ]:
all_important_features_xgb = set()

for col in col_list:
    features = locals()[f'features_{col}']
    target = locals()[f'close_{col}']['Close']
    features = pd.DataFrame(StandardScaler().fit_transform(features),
                                   columns=features.columns,
                                   index=features.index)

    xgb = XGBRegressor(n_estimators=100, random_state=42)
    xgb.fit(features, target)

    xgb_importance = pd.DataFrame({
        'Feature': features.columns,
        'Importance': xgb.feature_importances_
    }).sort_values('Importance', ascending=False)

    xgb_features = xgb_importance[xgb_importance['Importance'] > threshold]['Feature'].tolist()
    all_important_features_xgb.update(xgb_features)
    print(f"XGBoost selected {len(xgb_features)} features for {col}:",xgb_features)

print("\n=== Final XGBoost Important Features ===")
print(f"Total unique features: {len(all_important_features_xgb)}")
print(sorted(all_important_features_xgb))

XGBoost selected 1 features for DGS1MO: ['f2']
XGBoost selected 2 features for DGS3MO: ['f19', 'f2']
XGBoost selected 1 features for DGS6MO: ['f2']
XGBoost selected 1 features for DGS1: ['f2']
XGBoost selected 1 features for DGS2: ['f2']
XGBoost selected 1 features for DGS3: ['f2']
XGBoost selected 1 features for DGS5: ['f2']
XGBoost selected 1 features for DGS7: ['f2']
XGBoost selected 1 features for DGS10: ['f2']
XGBoost selected 2 features for DGS20: ['f2', 'f19']
XGBoost selected 1 features for DGS30: ['f2']

=== Final XGBoost Important Features ===
Total unique features: 2
['f19', 'f2']


Combined Results

In [ ]:
final_features = all_important_features_ridge.union(all_important_features_gb).union(all_important_features_xgb)
print("\n=== All Unique Important Features Across Methods ===")
print(f"Total unique features: {len(final_features)}")
print(sorted(final_features))


=== All Unique Important Features Across Methods ===
Total unique features: 14
['f10', 'f11', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f2', 'f3', 'f4', 'f6', 'f7']


In [ ]:
final_features

{'f10',
 'f11',
 'f13',
 'f14',
 'f15',
 'f16',
 'f17',
 'f18',
 'f19',
 'f2',
 'f3',
 'f4',
 'f6',
 'f7'}

In [ ]:
with open('Technical Factors.txt', 'w') as f:
    for feature in final_features:
        f.write(feature + '\n')